# Word2Vec (CBOW) — Nearest Words with t-SNE Visualization and FAISS HNSW Search

**Pipeline**
1. Train a Word2Vec model using the **CBOW** architecture on a large text corpus.
2. Visualize the learned embeddings in 2D using **t-SNE**.
3. Build a **FAISS HNSW** index on the embeddings and retrieve the **3 nearest neighbors** for every word, then evaluate the approximate search against an exact (brute-force) baseline.
4. Print the **10 most similar word pairs** found across the whole vocabulary.


## 0. Setup
Install the libraries we need. `gensim` gives us Word2Vec, `faiss-cpu` gives us the HNSW approximate nearest-neighbor index.

In [ ]:
!pip install -q gensim faiss-cpu scikit-learn matplotlib adjustText


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from gensim.models import Word2Vec

from sklearn.manifold import TSNE
from adjustText import adjust_text
from matplotlib.patches import Ellipse

import faiss

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## Step 1 — Train Word2Vec with CBOW

Instead of a pre-packaged dataset, we build our own corpus from 8 full novels/novellas downloaded from
**Project Gutenberg** (public domain, plain UTF-8 text):

- *Crime and Punishment* — Fyodor Dostoevsky
- *The Brothers Karamazov* — Fyodor Dostoevsky
- *Moby Dick* — Herman Melville
- *Dracula* — Bram Stoker
- *Frankenstein* — Mary Shelley
- *The Trial* — Franz Kafka
- *Alice's Adventures in Wonderland* — Lewis Carroll
- *Through the Looking-Glass* — Lewis Carroll

Combined, this gives roughly **1.1 million tokens**. With `min_count=5` the model keeps a vocabulary of
about **10,000 words**, which is large enough for meaningful nearest-neighbour relationships and small
enough to train on Colab in under a minute.

The cell below downloads each book, strips the Project Gutenberg license header/footer boilerplate,
cleans and tokenizes the text, and splits it into fixed-size chunks (so the rest of the pipeline sees the
same `sentences` format — a list of token lists — regardless of which corpus we use).

In [ ]:
import urllib.request
import re
from gensim.utils import simple_preprocess

# The 8-book corpus chosen for this project.
GUTENBERG_BOOKS = {
    2554: "Crime and Punishment - Fyodor Dostoevsky",
    28054: "The Brothers Karamazov - Fyodor Dostoevsky",
    2701: "Moby Dick - Herman Melville",
    345: "Dracula - Bram Stoker",
    84: "Frankenstein - Mary Shelley",
    7849: "The Trial - Franz Kafka",
    11: "Alice's Adventures in Wonderland - Lewis Carroll",
    12: "Through the Looking-Glass - Lewis Carroll",
}

GUTENBERG_START_RE = re.compile(r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.IGNORECASE | re.DOTALL)
GUTENBERG_END_RE = re.compile(r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*", re.IGNORECASE | re.DOTALL)

def download_book(book_id: int) -> str:
    """Try a couple of common Gutenberg URL patterns and return the raw text."""
    urls = [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
    ]
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=20) as resp:
                return resp.read().decode("utf-8", errors="ignore")
        except Exception:
            continue
    raise RuntimeError(f"Could not download book id {book_id} from any known URL pattern")

def strip_gutenberg_boilerplate(text: str) -> str:
    """Remove the standard Project Gutenberg license header and footer."""
    text = GUTENBERG_START_RE.split(text, maxsplit=1)[-1]
    text = GUTENBERG_END_RE.split(text, maxsplit=1)[0]
    return text

raw_texts = []
for book_id, title in GUTENBERG_BOOKS.items():
    try:
        raw = download_book(book_id)
        cleaned = strip_gutenberg_boilerplate(raw)
        raw_texts.append(cleaned)
        print(f"Downloaded: {title} ({len(cleaned):,} characters)")
    except Exception as e:
        print(f"Skipping '{title}' (id {book_id}): {e}")

full_text = "\n".join(raw_texts)
print(f"\nTotal corpus size: {len(full_text):,} characters from {len(raw_texts)} books")


In [ ]:
# Tokenize the whole corpus (lowercases, strips punctuation, drops very short tokens)
all_tokens = simple_preprocess(full_text, deacc=True, min_len=2)

# Split into fixed-size pseudo-sentences, matching the chunking style of the text8 dataset,
# so downstream Word2Vec training sees the same "list of token-list" format.
CHUNK_SIZE = 1000
sentences = [all_tokens[i:i + CHUNK_SIZE] for i in range(0, len(all_tokens), CHUNK_SIZE)]

n_tokens = len(all_tokens)
print(f"Number of chunks (pseudo-sentences): {len(sentences):,}")
print(f"Total number of tokens: {n_tokens:,}")
print("Example tokens:", sentences[0][:20])


In [ ]:
# Train Word2Vec using the CBOW architecture (sg=0 => CBOW, sg=1 would be Skip-gram)
start = time.time()

model = Word2Vec(
    sentences=sentences,
    vector_size=100,     # embedding dimensionality
    window=5,             # context window size
    min_count=5,          # ignore words that appear fewer than 5 times
    workers=4,             # parallel training threads
    sg=0,                 # CBOW (Continuous Bag Of Words)
    epochs=15,
    seed=RANDOM_SEED,
)

print(f"Training took {time.time() - start:.1f} seconds")
print(f"Vocabulary size: {len(model.wv.index_to_key):,}")


In [ ]:
# Save the trained model so it can be reloaded without retraining
model.save("word2vec_cbow_gutenberg.model")

# Quick sanity check: does the model learn sensible relationships?
test_words = [
    "alice",
    "rabbit",
    "queen",
    "captain",
    "sea",
    "crime",
    "death",
    "man",
    "woman",
    "king"
]

for word in test_words:
    if word in model.wv:
        print(f"\n{word}")
        for neighbor, score in model.wv.most_similar(word, topn=5):
          print(f"{neighbor:<15} {score:.3f}")

## Step 2 — Dimensionality Reduction with t-SNE

The embeddings live in a 100-dimensional space, which we cannot plot directly. We reduce them to 2D with
**t-SNE** so we can visualize how words group together. For readability we only plot the N most frequent
words (plotting the whole vocabulary would be an unreadable blob).

**Clustering is done in the original 100-d space, not on the t-SNE output.** t-SNE preserves local
neighbourhoods but distorts global distances and cluster sizes, so clusters found on its 2-D coordinates
partly reflect t-SNE's own layout. Instead we L2-normalise the embeddings (so Euclidean KMeans behaves like
cosine-similarity clustering), pick K by silhouette score in that space, and only *colour* the t-SNE plot
with the resulting labels. Silhouette scores in 100 dimensions are naturally much lower than on a 2-D
projection; what matters is the relative comparison between values of K.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

N_WORDS_TO_PLOT = 200

top_words = model.wv.index_to_key[:N_WORDS_TO_PLOT]
top_vectors = np.array([model.wv[w] for w in top_words])
top_vectors_unit = normalize(top_vectors)  # unit length -> Euclidean KMeans ~ cosine clustering

# ---- Choose K by silhouette score in the 100-d embedding space ----
K_RANGE = range(3, 13)
silhouettes = {}
for k in K_RANGE:
    labels = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10).fit_predict(top_vectors_unit)
    silhouettes[k] = silhouette_score(top_vectors_unit, labels, metric="cosine")

N_CLUSTERS = max(silhouettes, key=silhouettes.get)
clusters = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10).fit_predict(top_vectors_unit)

print("Silhouette (cosine) by K:", {k: round(v, 3) for k, v in silhouettes.items()})
print(f"Best K = {N_CLUSTERS} (silhouette = {silhouettes[N_CLUSTERS]:.3f})")

for c in range(N_CLUSTERS):
    members = [w for w, lab in zip(top_words, clusters) if lab == c]
    print(f"  cluster {c}: {', '.join(members[:12])}{' ...' if len(members) > 12 else ''}")

# ---- t-SNE for display only ----
print(f"\nReducing {len(top_words)} words to 2D with t-SNE...")
tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30, init="pca",
            learning_rate="auto", metric="cosine")
tsne_result = tsne.fit_transform(top_vectors)

In [ ]:
plt.figure(figsize=(16, 14))

colors = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS))

# ----------------------------
# Draw soft cluster ellipses
# ----------------------------

for c in range(N_CLUSTERS):

    pts = tsne_result[clusters == c]

    center = pts.mean(axis=0)

    cov = np.cov(pts.T)

    eigvals, eigvecs = np.linalg.eigh(cov)

    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    angle = np.degrees(np.arctan2(*eigvecs[:,0][::-1]))

    scale = 2.2   # <-- controls how "big" the ellipse is

    width = 2 * scale * np.sqrt(eigvals[0])
    height = 2 * scale * np.sqrt(eigvals[1])

    ellipse = Ellipse(
        center,
        width,
        height,
        angle=angle,
        facecolor=colors[c],
        edgecolor=colors[c],
        alpha=0.10,
        linewidth=2
    )

    plt.gca().add_patch(ellipse)


# ----------------------------
# Plot points
# ----------------------------
for c in range(N_CLUSTERS):

    pts = tsne_result[clusters == c]

    plt.scatter(
        pts[:,0],
        pts[:,1],
        s=160,
        color=colors[c],
        edgecolor='white',
        linewidth=1.2,
        alpha=0.9,
        zorder=2
    )

# ----------------------------
# Labels
# ----------------------------
texts = []

for i, word in enumerate(top_words):
    texts.append(
        plt.text(
            tsne_result[i,0],
            tsne_result[i,1],
            word,
            fontsize=10,
            color="black",
            zorder=3
        )
    )

# Automatically move overlapping labels
adjust_text(
    texts,
    expand_points=(2.0, 2.0),
    expand_text=(1.8, 1.8),
    force_points=0.8,
    force_text=1.2,
    lim=500,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.3,
        alpha=0.35
    )
)

plt.title(f"CBOW embeddings, t-SNE projection (colours: KMeans in 100-d space, K={N_CLUSTERS})", fontsize=18)
plt.xlabel("t-SNE dimension 1")
plt.ylabel("t-SNE dimension 2")

plt.tight_layout()
plt.savefig("tsne_word_embeddings.png", dpi=300)
plt.show()

## Step 3 — Nearest Words with FAISS HNSW

We now search the full embedding space (not just the top 200 plotted words) for the 3 closest words to every word in the vocabulary, using the **HNSW** (Hierarchical Navigable Small World) index from **FAISS**.

HNSW is an approximate nearest neighbor algorithm built as a multi-layer navigable graph, which lets it find near-neighbors in large embedding spaces far faster than scanning every vector. We build the index, run the search for all words, and evaluate the results below: how fast the search ran, and how good/confident the matches it returned actually are.

We use cosine similarity, implemented as inner product on L2-normalized vectors.

**The HNSW search is performed for every word embedding in the vocabulary, producing the three nearest neighbors for each word. Since printing the results for thousands of words would be impractical, only a small set of representative examples is displayed below, while the complete results are stored and used in the following evaluation and ranking steps.**



In [ ]:
# Use the FULL vocabulary learned by Word2Vec (not just the 200 words used for the t-SNE plot),
# so the nearest-neighbor search and its performance evaluation reflect the whole embedding space.
vocab = model.wv.index_to_key
vectors = model.wv.vectors.astype("float32")

n_words, dim = vectors.shape
print(f"Building indexes over the full vocabulary: {n_words:,} words, dimension {dim}")

# Normalize vectors so inner product equals cosine similarity
normed_vectors = vectors.copy()
faiss.normalize_L2(normed_vectors)


In [ ]:
# ---- HNSW index ----
M = 32                 # number of neighbors per node in the HNSW graph
index_hnsw = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
index_hnsw.hnsw.efConstruction = 200   # quality of graph construction
index_hnsw.hnsw.efSearch = 128         # quality/speed trade-off at search time

start = time.time()
index_hnsw.add(normed_vectors)
build_time_hnsw = time.time() - start

print(f"HNSW index built in {build_time_hnsw:.2f}s over {n_words:,} words")


In [ ]:
K = 4  # we ask for 4 because the closest match to a word is always itself; we drop it afterwards

start = time.time()
D_hnsw, I_hnsw = index_hnsw.search(normed_vectors, K)
search_time_hnsw = time.time() - start

print(f"HNSW search over {n_words:,} words took {search_time_hnsw:.3f}s")
print(f"Average search time per word: {(search_time_hnsw / n_words) * 1000:.4f} ms")


In [ ]:
def top3_neighbors(I, D, idx):
    """Return the top-3 nearest neighbors for a word, excluding the word itself."""

    neighbors = []

    for neighbor, score in zip(I[idx], D[idx]):
        if neighbor != idx:
            neighbors.append((neighbor, float(score)))

        if len(neighbors) == 3:
            break

    neighbor_idx = [n for n, _ in neighbors]
    neighbor_scores = [s for _, s in neighbors]

    return neighbor_idx, neighbor_scores

# Build a {word: [(neighbor_word, cosine_similarity), ...]} dictionary using the HNSW results
nearest_words_hnsw = {}
for i, word in enumerate(vocab):
    n_idx, n_scores = top3_neighbors(I_hnsw, D_hnsw, i)
    nearest_words_hnsw[word] = [(vocab[j], float(s)) for j, s in zip(n_idx, n_scores)]

print("Sample nearest-neighbor search results (randomly selected words):\n")

rng = np.random.default_rng(RANDOM_SEED)

# Randomly choose 6 words from the 1,000 most frequent words
sample_words = rng.choice(vocab[:1000], size=6, replace=False)

for w in sample_words:

    print(f"{w}:")

    for neighbor, score in nearest_words_hnsw[w]:
        print(f"   {neighbor:<15} {score:.3f}")

    print()


### Performance evaluation

HNSW is an *approximate* search, so the main question is how close its answers are to the exact answer
and how much time it saves. We evaluate it on three axes:

1. **Accuracy vs. exact search** — build a brute-force `IndexFlatIP` over the same normalised vectors
   (exact cosine k-NN) and measure **recall@3**: the fraction of each word's true 3 nearest neighbours
   that HNSW also returned.
2. **Speed** — index build time, total query time, and speed-up over brute force.
3. **Speed/accuracy trade-off** — sweep HNSW's `efSearch` parameter and record recall and query time.

We also report the distribution of top-1 cosine similarities. That number describes the *embeddings*
(how tight each word's closest neighbour is), not the search algorithm, so it is reported separately.

In [ ]:
# ---- Exact (brute-force) baseline over the same normalised vectors ----
index_flat = faiss.IndexFlatIP(dim)
index_flat.add(normed_vectors)

start = time.time()
D_flat, I_flat = index_flat.search(normed_vectors, K)
search_time_flat = time.time() - start

def recall_at_3(I_approx, I_exact):
    """Mean fraction of each word's exact top-3 neighbours (excluding itself) that HNSW also found."""
    hits = 0
    for i in range(len(I_exact)):
        exact = set(top3_neighbors(I_exact, D_flat, i)[0])
        approx = set(top3_neighbors(I_approx, D_hnsw, i)[0])
        hits += len(exact & approx)
    return hits / (3 * len(I_exact))

hnsw_recall = recall_at_3(I_hnsw, I_flat)
top1_agreement = np.mean([
    top3_neighbors(I_hnsw, D_hnsw, i)[0][:1] == top3_neighbors(I_flat, D_flat, i)[0][:1]
    for i in range(n_words)
])

print(f"Brute-force search time : {search_time_flat:.3f}s")
print(f"HNSW search time        : {search_time_hnsw:.3f}s  (speed-up x{search_time_flat / search_time_hnsw:.1f})")
print(f"HNSW recall@3           : {hnsw_recall:.4f}")
print(f"Top-1 neighbour agrees  : {top1_agreement * 100:.2f}% of words")

**Speed/accuracy trade-off.** `efSearch` is the size of the candidate list HNSW keeps while walking the
graph. Larger values explore more of the graph: higher recall, slower queries.

At ~10k vectors, brute force is already fast (it is one matrix multiplication), so the speed-up here is
modest. HNSW's advantage grows with the collection: brute-force cost rises linearly with the number of
vectors, while HNSW's query cost grows roughly logarithmically.

In [ ]:
sweep = []
for ef in [8, 16, 32, 64, 128, 256]:
    index_hnsw.hnsw.efSearch = ef
    start = time.time()
    D_tmp, I_tmp = index_hnsw.search(normed_vectors, K)
    elapsed = time.time() - start
    hits = sum(len(set(top3_neighbors(I_flat, D_flat, i)[0]) & set(top3_neighbors(I_tmp, D_tmp, i)[0]))
               for i in range(n_words))
    sweep.append((ef, hits / (3 * n_words), elapsed * 1000 / n_words))
index_hnsw.hnsw.efSearch = 128  # restore the value used for the main results

print(f"{'efSearch':>8} | {'recall@3':>8} | {'ms / query':>10}")
for ef, rec, ms in sweep:
    print(f"{ef:>8} | {rec:>8.4f} | {ms:>10.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([ms for _, _, ms in sweep], [rec for _, rec, _ in sweep], marker="o")
for ef, rec, ms in sweep:
    ax.annotate(f"ef={ef}", (ms, rec), textcoords="offset points", xytext=(5, -10), fontsize=8)
ax.set_xlabel("ms per query")
ax.set_ylabel("recall@3 vs exact search")
ax.set_title("HNSW speed/accuracy trade-off")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("hnsw_tradeoff.png", dpi=200)
plt.show()

In [ ]:
# ---- Speed ----
print("===== FAISS HNSW performance evaluation =====")
print(f"Vocabulary size searched     : {n_words:,} words")
print(f"Index build time             : {build_time_hnsw:.2f}s")
print(f"Total search time (all words): {search_time_hnsw:.3f}s")
print(f"Average search time per word : {(search_time_hnsw / n_words) * 1000:.4f} ms")
print(f"Words searched per second    : {n_words / search_time_hnsw:,.0f}")

# ---- Quality of matches ----
# Take each word's single closest neighbor's similarity score, and look at the
# distribution: are the matches HNSW is returning generally strong/confident,
# or weak/noisy?
top1_scores = np.array([
    nearest_words_hnsw[word][0][1]
    for word in vocab
    if nearest_words_hnsw[word]
])

CONFIDENCE_THRESHOLD = 0.5  # cosine similarity above which we call a match "confident"

print("\n===== Embedding tightness: top-1 cosine similarity per word =====")
print(f"Words with at least one neighbor : {len(top1_scores):,} / {n_words:,}")
print(f"Mean top-1 similarity            : {top1_scores.mean():.3f}")
print(f"Median top-1 similarity          : {np.median(top1_scores):.3f}")
print(f"Min / Max top-1 similarity       : {top1_scores.min():.3f} / {top1_scores.max():.3f}")
print(f"Words with top-1 similarity > {CONFIDENCE_THRESHOLD} : "
      f"{(top1_scores > CONFIDENCE_THRESHOLD).sum():,} "
      f"({(top1_scores > CONFIDENCE_THRESHOLD).mean() * 100:.1f}%)")

**Write-up**

- **Search accuracy.** Recall@3 against exact brute-force search (printed above) shows how many of the
  true nearest neighbours HNSW returns at `efSearch=128`. The sweep shows the usual trade-off: small
  `efSearch` values are faster but miss neighbours; recall saturates well before the largest values.
- **Speed.** On ~10k vectors brute force is a single fast matrix multiplication, so HNSW gives little or
  no speed-up here and can even be slower (the printed speed-up may be below 1). The point of HNSW is
  scaling: for millions of vectors exact search becomes
  the bottleneck while HNSW queries stay in the sub-millisecond range.
- **Embedding quality.** The examples below show where the embeddings work well (numbers, professions,
  character names that always co-occur) and where they are weaker (frequent, context-free words whose
  neighbours are diffuse). Note that a high cosine similarity means "used in similar contexts", not
  "synonym": names that always appear together, like *humpty*/*dumpty*, score highest of all.

In [ ]:
for word in ["six", "lawyer", "captain", "bit"]:
    if word in nearest_words_hnsw:
        neighbours = ", ".join(f"{w} ({s:.2f})" for w, s in nearest_words_hnsw[word])
        print(f"{word:<8} -> {neighbours}")

## Step 4 — Top 10 Most Similar Word Pairs

Finally, using the HNSW results from Step 3, we scan every word's single closest neighbor, rank all of
these (word, neighbor, similarity) triples by similarity, and print the **10 most similar pairs** overall.

Two things worth filtering for here:

- **True one-off noise vs. real rare collocations.** A word seen only once or twice can converge to a
  spuriously high similarity purely by chance. But some rare word pairs are genuinely, tightly linked —
  fixed phrases like "per cent" or characters like "Humpty"/"Dumpty" that are *always* adjacent. Rather
  than cutting off by frequency rank (which risks throwing out the real ones too), we filter by an actual
  minimum occurrence count, low enough to keep genuine rare collocations but high enough to drop one-off
  flukes.
- **Repetition.** Without care, a single very "clustered" word (e.g. a character constantly mentioned
  alongside their siblings) can occupy several of the 10 slots as the top match for multiple other words.
  We enforce that each word appears at most once across the Top 10, so the list reflects 10 *different*
  relationships rather than one dominant cluster repeated several times.


In [ ]:
from collections import Counter

MIN_WORD_COUNT = 8  # minimum number of times a word must appear in the corpus to be eligible here
word_counts = Counter(all_tokens)

pairs = []
for word, neighbors in nearest_words_hnsw.items():
    if word_counts[word] < MIN_WORD_COUNT:
        continue
    if neighbors:
        best_neighbor, best_score = neighbors[0]
        if word_counts[best_neighbor] < MIN_WORD_COUNT:
            continue
        pairs.append((word, best_neighbor, best_score))

# Sort by similarity, descending
pairs.sort(key=lambda x: x[2], reverse=True)

seen_pairs = set()
used_words = set()
top_pairs = []
for w1, w2, score in pairs:
    key = frozenset((w1, w2))
    if key in seen_pairs:
        continue
    if w1 in used_words or w2 in used_words:
        continue  # enforce variety: each word appears in at most one pair
    seen_pairs.add(key)
    used_words.add(w1)
    used_words.add(w2)
    top_pairs.append((w1, w2, score))
    if len(top_pairs) == 10:
        break

print("Top 10 most similar word pairs (CBOW Word2Vec + FAISS HNSW):\n")
for rank, (w1, w2, score) in enumerate(top_pairs, start=1):
    print(f"{rank:2d}. {w1:15s} <-> {w2:15s}   cosine similarity = {score:.3f}")
